In [1]:
import numpy as np
import xarray as xr
import time
import os
import glob
from typing import Tuple, List, Dict
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
from scipy import signal, fft
import cmaps
import sys

# 导入wave_tools中的交叉谱函数
sys.path.insert(0, '/work/mh1498/m301257')
from wave_tools import calculate_cross_spectrum, quick_cross_spectrum

FIG_SAVE_DIR = "../figures/cross_spectrum_specific_layer2/"

# 创建必要的目录
for d in [FIG_SAVE_DIR]:
    os.makedirs(d, exist_ok=True)

EXPERIMENTS = ['CNTL', 'P4K', '4CO2']

print("✅ 导入完成！使用wave_tools.calculate_cross_spectrum进行交叉谱计算")

✅ 导入完成！使用wave_tools.calculate_cross_spectrum进行交叉谱计算


In [2]:
# ============ 注释掉原有的函数定义，现在使用wave_tools模块 ============

# 原函数已经移动到 wave_tools/cross_spectrum.py
# 可以通过以下方式导入使用：
#   from wave_tools import calculate_cross_spectrum
#   result = calculate_cross_spectrum(X, Y, segLen=96, segOverLap=-65)

print("✅ 交叉谱函数已从wave_tools模块导入！")
print("   主要函数: calculate_cross_spectrum, quick_cross_spectrum")
print("   配置类: CrossSpectrumConfig")
print("   工具函数: remove_annual_cycle, nan_to_value_by_interp_3D")

✅ 交叉谱函数已从wave_tools模块导入！
   主要函数: calculate_cross_spectrum, quick_cross_spectrum
   配置类: CrossSpectrumConfig
   工具函数: remove_annual_cycle, nan_to_value_by_interp_3D


In [14]:
# ============ 加载降水数据 ============

def load_precipitation_data(experiment='cntl', data_dir='/work/mh1498/m301257/data_origin'):
    """
    加载降水数据
    /work/mh1498/m301257/processed_data/pr_4co2_2deg_interp.nc
    Parameters:
    -----------
    experiment : str
        实验名称，可选: 'cntl', 'p4k', '4co2'
    data_dir : str
        数据目录
    
    Returns:
    --------
    xr.DataArray
        降水数据
    """
    # 构建文件路径
    file_path = os.path.join(data_dir, f'pr_{experiment}_2deg_interp.nc')
    
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"未找到实验 {experiment} 的降水数据文件: {file_path}")
    
    print(f"📂 加载数据: {file_path}")
    
    # 加载数据 - 使用 open_dataarray 因为数据是以 DataArray 形式保存的
    try:
        pr = xr.open_dataarray(file_path, engine='netcdf4')
    except:
        try:
            pr = xr.open_dataarray(file_path, engine='h5netcdf')
        except:
            pr = xr.open_dataarray(file_path)
    
    print(f"✅ 数据加载成功")
    print(f"   形状: {pr.shape}")
    print(f"   维度: {pr.dims}")
    print(f"   时间范围: {pr.time.values[0]} to {pr.time.values[-1]}")
    
    return pr


def load_all_precipitation_data():
    """
    加载所有实验的降水数据
    
    Returns:
    --------
    dict
        包含所有实验降水数据的字典
    """
    print("\n" + "="*60)
    print("加载所有降水数据")
    print("="*60)
    
    experiments = ['cntl', 'p4k', '4co2']
    pr_data = {}
    
    for exp in experiments:
        try:
            print(f"\n加载 {exp.upper()}...")
            pr = load_precipitation_data(exp)
            pr_data[exp] = pr  # 直接使用原始单位 (kg m^-2 s^-1)
        except Exception as e:
            print(f"❌ 错误: {e}")
    
    print(f"\n{'='*60}")
    print(f"✅ 成功加载 {len(pr_data)} 个实验的数据")
    print(f"{'='*60}")
    
    return pr_data




In [15]:
# ============ 快速加载多层散度数据 ============

def load_divergence_by_level(experiment='cntl', level=80, data_dir='/work/mh1498/m301257/3D_data/divergence'):
    """
    快速加载指定层次的散度数据
    
    Parameters:
    -----------
    experiment : str
        实验名称，可选: 'cntl', 'p4k', '4co2'
    level : int
        层次编号，例如: 80, 81, 55, 50
    data_dir : str
        数据目录
    
    Returns:
    --------
    xr.DataArray
        散度数据
    
    Example:
    --------
    >>> div_cntl_lev80 = load_divergence_by_level('cntl', 80)
    >>> div_p4k_lev55 = load_divergence_by_level('p4k', 55)
    
    /work/mh1498/m301257/processed_data/divergence_lev80_cntl.nc
    """
    # 构建文件路径
    file_pattern = os.path.join(data_dir, f'divergence_lev{level}_{experiment}.nc')
    files = glob.glob(file_pattern)
    
    if not files:
        raise FileNotFoundError(f"未找到实验 {experiment} Level {level} 的散度数据文件: {file_pattern}")
    
    if len(files) > 1:
        print(f"⚠️ 找到多个文件，使用第一个: {files[0]}")
    
    file_path = files[0]
    print(f"📂 加载数据: {file_path}")
    
    # 加载数据 - 尝试不同的引擎
    try:
        ds = xr.open_dataset(file_path, engine='netcdf4')
    except:
        try:
            ds = xr.open_dataset(file_path, engine='h5netcdf')
        except:
            # 如果都失败，尝试默认引擎
            ds = xr.open_dataset(file_path)
    
    div = ds['divergence']
    
    print(f"✅ 数据加载成功")
    print(f"   形状: {div.shape}")
    print(f"   维度: {div.dims}")
    print(f"   时间范围: {div.time.values[0]} to {div.time.values[-1]}")
    print(f"   空间范围: lat [{div.lat.min().values:.1f}, {div.lat.max().values:.1f}], "
          f"lon [{div.lon.min().values:.1f}, {div.lon.max().values:.1f}]")
    
    return div




In [5]:
# ============ 网格对齐函数 ============

def align_grids(pr_data, div_data):
    """
    确保降水和散度数据在相同的网格上
    
    Parameters:
    -----------
    pr_data : xr.DataArray
        降水数据
    div_data : xr.DataArray
        散度数据
    
    Returns:
    --------
    tuple : (pr_aligned, div_aligned)
        对齐后的降水和散度数据
    """
    print(f"  原始形状: pr={pr_data.shape}, div={div_data.shape}")
    
    # 检查经度维度
    pr_lon = pr_data.lon.values
    div_lon = div_data.lon.values
    
    if len(pr_lon) != len(div_lon):
        print(f"  ⚠️ 经度维度不匹配: pr有{len(pr_lon)}个点, div有{len(div_lon)}个点")
        print(f"  🔧 将散度数据插值到降水网格...")
        
        # 将散度数据插值到降水的网格，使用最近邻以避免边界NaN
        div_aligned = div_data.interp(lon=pr_data.lon, lat=pr_data.lat, 
                                      method='linear', 
                                      kwargs={'fill_value': 'extrapolate'})
        
        # 检查并处理NaN值
        nan_count_before = np.isnan(div_aligned.values).sum()
        if nan_count_before > 0:
            print(f"  ⚠️ 插值后发现 {nan_count_before} 个NaN值，进行填充...")
            # 使用最近邻填充剩余的NaN
            div_aligned = div_aligned.fillna(div_data.interp(lon=pr_data.lon, lat=pr_data.lat, 
                                                              method='nearest'))
            nan_count_after = np.isnan(div_aligned.values).sum()
            print(f"  ✓ 填充后剩余 {nan_count_after} 个NaN值")
            
            # 如果还有NaN，用0填充（极端情况）
            if nan_count_after > 0:
                div_aligned = div_aligned.fillna(0)
                print(f"  ✓ 剩余NaN值已用0填充")
    else:
        div_aligned = div_data
    
    print(f"  对齐后形状: pr={pr_data.shape}, div={div_aligned.shape}")
    
    return pr_data, div_aligned

In [16]:

div_cntl = load_divergence_by_level('cntl', 81)  # 使用Level 80 (约500 hPa)
div_p4k = load_divergence_by_level('p4k', 81)
div_4co2 = load_divergence_by_level('4co2', 81)
# 加载所有实验的降水数据
all_pr = load_all_precipitation_data()
all_div = {'cntl': div_cntl, 'p4k': div_p4k, '4co2': div_4co2}

📂 加载数据: /work/mh1498/m301257/3D_data/divergence/divergence_lev81_cntl.nc
✅ 数据加载成功
   形状: (5114, 15, 180)
   维度: ('time', 'lat', 'lon')
   时间范围: 1980-01-01T00:00:00.000000000 to 1993-12-31T00:00:00.000000000
   空间范围: lat [-14.0, 14.0], lon [0.0, 358.0]
📂 加载数据: /work/mh1498/m301257/3D_data/divergence/divergence_lev81_p4k.nc
✅ 数据加载成功
   形状: (5114, 15, 180)
   维度: ('time', 'lat', 'lon')
   时间范围: 1980-01-01T00:00:00.000000000 to 1993-12-31T00:00:00.000000000
   空间范围: lat [-14.0, 14.0], lon [0.0, 358.0]
📂 加载数据: /work/mh1498/m301257/3D_data/divergence/divergence_lev81_4co2.nc
✅ 数据加载成功
   形状: (5114, 15, 180)
   维度: ('time', 'lat', 'lon')
   时间范围: 1980-01-01T00:00:00.000000000 to 1993-12-31T00:00:00.000000000
   空间范围: lat [-14.0, 14.0], lon [0.0, 358.0]

加载所有降水数据

加载 CNTL...
📂 加载数据: /work/mh1498/m301257/data_origin/pr_cntl_2deg_interp.nc
✅ 数据加载成功
   形状: (5114, 15, 180)
   维度: ('time', 'lat', 'lon')
   时间范围: 1980-01-01T00:00:00.000000000 to 1993-12-31T00:00:00.000000000

加载 P4K...
📂 加载数据: /work/

In [7]:
# ============ 定义CCKW波段函数 ============

def get_curve():
    """
    计算Kelvin波的色散曲线和CCKW波段
    """
    # 地球半径 (m)
    re = 6371 * 1000
    
    # 最大频率
    fmax = np.array([1/3, 1/2.25, 0.5])
    
    # 重力加速度 (m/s^2)
    g = 9.8
    s2d = 86400  # 秒到天的转换
    
    # 准备CCKW波段
    kw_x = []
    kw_y = []
    
    for v in range(3):  # 对每个深度类别
        he = [8, 25, 90] 
        s_min = (g * 8) ** 0.5 / (2 * np.pi * re) * s2d  # 最小斜率
        s_max = (g * 90) ** 0.5 / (2 * np.pi * re) * s2d  # 最大斜率

        kw_tmax = 20

        kw_x.append(np.array([
            2,
            1 / kw_tmax / s_min,
            14,
            14,
            fmax[0] / s_max,
            2,
            2,
        ]))
        
        kw_y.append(np.array([
            1 / kw_tmax,
            1 / kw_tmax,
            14 * s_min,
            fmax[0],
            fmax[0],
            2 * s_max,
            1/20,
        ]))
    
    return kw_x, kw_y

kw_x, kw_y = get_curve()
print("✅ CCKW波段计算完成！")

✅ CCKW波段计算完成！


In [8]:
# # ============ 绘制三个实验的交叉谱对比图 ============

# s2d = 86400
# re = 6371 * 1000

# # 创建3个子图，横向排列
# fig, axes = plt.subplots(1, 3, figsize=(16, 8), dpi=300)
# plt.subplots_adjust(left=0.06, right=0.98, top=0.92, bottom=0.15, wspace=0.25)
# plt.rcParams.update({'font.size': 10})

# # 实验名称和标题
# exp_names = ['cntl', 'p4k', '4co2']
# exp_titles = ['CNTL', 'P4K', '4CO2']

# # 设置coherence²阈值 - 只在显著区域显示箭头
# # 使用95%置信水平的阈值
# coh2_threshold = 0.05  # 可以根据显著性检验调整

# for idx, (exp_name, exp_title, ax) in enumerate(zip(exp_names, exp_titles, axes)):
#     plt.sca(ax)
    
#     # 获取当前实验的结果
#     stc = results[exp_name]['STC']
#     zonalwnum = results[exp_name]['wave']
#     freq = results[exp_name]['freq']
    
#     # 绘制coherence squared的等高线图
#     levels = np.linspace(0.1, 0.4, 31)
#     contourf = stc[4].plot.contourf(
#         ax=ax,
#         cmap=cmaps.WhiteBlueGreenYellowRed,
#         levels=21,
#         add_colorbar=False,
#         add_labels=False,
#         extend='neither'
#     )
    
#     # 准备矢量场数据 - 只在coherence²高的区域显示
#     # 增加采样间隔，减少箭头密度
#     skip = 2 # 每隔3个点取一个
    
#     # 提取数据
#     wave_sub = zonalwnum[::skip]
#     freq_sub = freq[::skip]
#     u_sub = stc[6][::skip, ::skip].values
#     v_sub = stc[7][::skip, ::skip].values
#     coh2_sub = stc[4][::skip, ::skip].values
    
#     # 创建掩蔽：只在coherence²超过阈值的地方显示箭头
#     mask = coh2_sub < coh2_threshold
#     u_masked = np.where(mask, np.nan, u_sub)
#     v_masked = np.where(mask, np.nan, v_sub)
    
#     # 添加矢量场（相位信息）
#     q = ax.quiver(
#         wave_sub, freq_sub,
#         u_masked, v_masked,
#         scale=30, headwidth=4, headlength=5, 
#         width=0.004, alpha=0.8
#     )
    
#     # 设置标题
#     ax.set_title(f'({chr(97 + idx)}) {exp_title}', fontsize=18,  loc='left')
#     ax.set_title('Symm', fontsize=10, loc='right')
    
#     # 设置坐标轴
#     ax.set_ylabel('Frequency (1/day)', fontsize=18)
#     ax.set_xlabel('Zonal wavenumber', fontsize=18)
#     ax.set_xlim(-15, 15)
#     ax.set_ylim(0, 0.5)
    
#     # 绘制CCKW波段
#     ax.plot(kw_x[0], kw_y[0], 'purple', linewidth=1.5, linestyle='solid', label='CCKW band')
    
#     # 添加参考线
#     d = np.array([3, 6, 20])
#     dname = ['3d', '6d', '20d']
#     he_all = np.array([8, 25, 90])
#     cp = (9.8 * he_all) ** 0.5
#     zwnum_goal = 0.5 / s2d / cp * 2 * np.pi * re
    
#     # 零波数线
#     ax.plot([0, 0], [0, 0.5], 'k', linewidth=1, linestyle=':')
    
#     # 周期线
#     for dd in range(len(d)):
#         ax.plot([-15, 15], [1/d[dd], 1/d[dd]], 'k', linewidth=1, linestyle=':')
#         ax.text(-14.8, 1/d[dd] + 0.01, dname[dd], fontsize=15, color='k')
    
#     # Kelvin波色散关系
#     for hh in range(len(he_all)):
#         ax.plot([0, zwnum_goal[hh]], [0, 0.5], 'grey', linewidth=1, linestyle='dashed')
    
#     # 添加"kelvin"标签
#     ax.text(12, 0.35, 'kelvin', ha="center", va="center", size=9,
#             bbox={'facecolor': 'w', 'alpha': 0.9, 'edgecolor': 'none'})
#     ax.tick_params(labelsize=18, which='both', top=True, right=True)
# # 添加颜色条
# cbar = fig.colorbar(contourf, ax=axes, orientation='horizontal', 
#                      pad=0.15, aspect=40, shrink=0.8)
# cbar.set_label('Coherence squared', fontsize=11)



# # 保存图片
# save_path = os.path.join(FIG_SAVE_DIR, 'cross_spectrum_.png')
# plt.savefig(save_path, dpi=300, bbox_inches='tight')
# print(f"\n✅ 图片已保存: {save_path}")

# plt.show()

In [9]:
# # ============ 绘制改进版：只显示显著区域的相位箭头 ============

# s2d = 86400
# re = 6371 * 1000

# # 创建3个子图，横向排列
# fig, axes = plt.subplots(1, 3, figsize=(16, 5.5), dpi=300)
# plt.subplots_adjust(left=0.06, right=0.98, top=0.92, bottom=0.15, wspace=0.25)
# plt.rcParams.update({'font.size': 10})

# # 实验名称和标题
# exp_names = ['cntl', 'p4k', '4co2']
# exp_titles = ['CNTL', 'P4K', '4CO2']

# # 计算95%显著性水平的coherence²阈值
# # 使用第一个实验的自由度（所有实验应该相同）
# dof = results['cntl']['dof']
# prob_coh2 = results['cntl']['prob_coh2']
# # 找到95%置信水平对应的阈值
# coh2_95 = prob_coh2[4]  # 对应95%置信水平
# print(f"95%显著性水平的coherence²阈值: {coh2_95:.4f}")
# print(f"自由度: {dof:.2f}")

# for idx, (exp_name, exp_title, ax) in enumerate(zip(exp_names, exp_titles, axes)):
#     plt.sca(ax)
    
#     # 获取当前实验的结果
#     stc = results[exp_name]['STC']
#     zonalwnum = results[exp_name]['wave']
#     freq = results[exp_name]['freq']
    
#     # 绘制coherence squared的等高线图
#     levels = np.linspace(0.1, 0.6, 11)
#     contourf = stc[4].plot.contourf(
#         ax=ax,
#         cmap=cmaps.MPL_BuPu,
#         levels=21,
#         add_colorbar=False,
#         add_labels=False,
#         extend='neither'
#     )
    
#     # 添加95%显著性等值线
#     cs = stc[4].plot.contour(
#         ax=ax,
#         levels=[coh2_95],
#         colors='red',
#         linewidths=2,
#         linestyles='solid',
#         add_labels=False
#     )
    
#     # 准备矢量场数据 - 只在coherence²高于95%显著性水平的区域显示
#     skip = 4  # 每隔4个点取一个，减少箭头密度
    
#     # 提取数据
#     wave_sub = zonalwnum[::skip]
#     freq_sub = freq[::skip]
#     u_sub = stc[6][::skip, ::skip].values
#     v_sub = stc[7][::skip, ::skip].values
#     coh2_sub = stc[4][::skip, ::skip].values
    
#     # 创建掩蔽：只在coherence²超过95%显著性水平的地方显示箭头
#     mask = coh2_sub < coh2_95
#     u_masked = np.where(mask, np.nan, u_sub)
#     v_masked = np.where(mask, np.nan, v_sub)
    
#     # 添加矢量场（相位信息）
#     q = ax.quiver(
#         wave_sub, freq_sub,
#         u_masked, v_masked,
#         scale=20, headwidth=5, headlength=6, 
#         width=0.004, alpha=0.9, color='black'
#     )
    
#     # 设置标题
#     ax.set_title(f'{exp_title}', fontsize=12, fontweight='bold', loc='left')
#     ax.set_title('Symm (95% sig.)', fontsize=10, loc='right')
    
#     # 设置坐标轴
#     ax.set_ylabel('Frequency (1/day)', fontsize=11)
#     ax.set_xlabel('Zonal wavenumber', fontsize=11)
#     ax.set_xlim(-15, 15)
#     ax.set_ylim(0, 0.5)
    
#     # 绘制CCKW波段
#     ax.plot(kw_x[0], kw_y[0], 'purple', linewidth=2, linestyle='solid', 
#             label='CCKW band', zorder=10)
    
#     # 添加参考线
#     d = np.array([3, 6, 20])
#     dname = ['3d', '6d', '20d']
#     he_all = np.array([8, 25, 90])
#     cp = (9.8 * he_all) ** 0.5
#     zwnum_goal = 0.5 / s2d / cp * 2 * np.pi * re
    
#     # 零波数线
#     ax.plot([0, 0], [0, 0.5], 'k', linewidth=1, linestyle=':', alpha=0.6)
    
#     # 周期线
#     for dd in range(len(d)):
#         ax.plot([-15, 15], [1/d[dd], 1/d[dd]], 'k', linewidth=0.8, 
#                 linestyle=':', alpha=0.6)
#         ax.text(-14.5, 1/d[dd] + 0.01, dname[dd], fontsize=8, alpha=0.7)
    
#     # Kelvin波色散关系
#     for hh in range(len(he_all)):
#         ax.plot([0, zwnum_goal[hh]], [0, 0.5], 'grey', linewidth=1.2, 
#                 linestyle='dashed', alpha=0.6)
    
#     # 添加"kelvin"标签
#     ax.text(11, 0.35, 'Kelvin', ha="center", va="center", size=9,
#             bbox={'facecolor': 'white', 'alpha': 0.8, 'edgecolor': 'gray', 
#                   'boxstyle': 'round,pad=0.3'})

# # 添加颜色条
# cbar = fig.colorbar(contourf, ax=axes, orientation='horizontal', 
#                      pad=0.12, aspect=40, shrink=0.8)



# # 保存图片
# # save_path = os.path.join(FIG_SAVE_DIR, 'cross_spectrum_comparison_with_significance.png')
# # plt.savefig(save_path, dpi=300, bbox_inches='tight')
# # print(f"\n✅ 改进版图片已保存: {save_path}")

# plt.show()

In [10]:
# ============ 获取所有可用的层次编号 ============

import re

data_dir = '/work/mh1498/m301257/3D_data/divergence/'

# 查找所有散度文件
div_files = glob.glob(os.path.join(data_dir, 'divergence_lev*_cntl.nc'))

# 提取层次编号
levels = []
for file in div_files:
    match = re.search(r'divergence_lev(\d+)_cntl\.nc', file)
    if match:
        levels.append(int(match.group(1)))

# 排序
levels = sorted(levels)

print(f"找到 {len(levels)} 个层次:")
print(f"层次列表: {levels}")
print(f"\n准备对每个层次进行交叉谱分析...")

找到 17 个层次:
层次列表: [41, 46, 51, 55, 58, 63, 67, 71, 74, 76, 78, 80, 81, 83, 84, 85, 87]

准备对每个层次进行交叉谱分析...


In [12]:
def _ocean(ds):
    fraction = xr.open_dataarray(r'../processed_data/land_mask_2deg.nc')
    return fraction == 0
ocean_mask = _ocean(None)

In [17]:
# ============ 循环处理所有层次 ============

# 设置去除年循环的参数
spd = 1  # samples per day
fCrit = 1.0 / 365.0  # 去除周期大于365天的变化

# 循环处理每个层次
for level in levels:
    print(f"\n{'='*80}")
    print(f"🔄 处理 Level {level}")
    print(f"{'='*80}")
    
    try:
        # ============ 步骤1: 加载当前层次的散度数据 ============
        print(f"\n📂 加载 Level {level} 的散度数据...")
        div_data = {}
        for exp_name in ['cntl', 'p4k', '4co2']:
            div_data[exp_name] = load_divergence_by_level(exp_name, level, data_dir)
        
        # ============ 步骤2: 计算交叉谱 ============
        print(f"\n📊 计算 Level {level} 的交叉谱...")
        level_results = {}
        
        for exp_name in ['cntl', 'p4k', '4co2']:
            print(f"\n  处理 {exp_name.upper()}...")
            
            # 获取降水和散度数据
            pr_raw = all_pr[exp_name]
            div_raw = div_data[exp_name]
            
            # 网格对齐 - 确保降水和散度在相同网格上
            pr_aligned, div_aligned = align_grids(pr_raw, div_raw)
            
            # 计算异常值（去除年循环）
            pr_ano = pr_aligned.groupby('time.dayofyear') - pr_aligned.groupby('time.dayofyear').mean()
            div_ano = div_aligned.groupby('time.dayofyear') - div_aligned.groupby('time.dayofyear').mean()
            
            # 应用海洋掩码并填充NaN（用0填充以避免交叉谱计算出错）
            pr_ano = pr_ano.where(ocean_mask, 0)
            div_ano = div_ano.where(ocean_mask, 0)
            
            # 计算交叉谱 - 使用wave_tools的函数
            result = calculate_cross_spectrum(
                pr_ano, div_ano, 
                segLen=96, 
                segOverLap=-65,
                symmetry='symm',
                return_xarray=True
            )
            
            # 提取结果（现在result['STC']已经是xarray DataArray）
            if result is None:
                print(f"    ✗ 交叉谱计算失败，返回None")
                continue
                
            stc_da = result['STC']  # 已经是xarray DataArray
            zonalwnum = result['wave']
            freq = result['freq']
            stc = stc_da.values  # 如果需要numpy数组
            
            # 存储结果 - stc_da已经是正确格式，不需要再次转换
            level_results[exp_name] = {
                'STC': stc_da,  # 直接使用返回的DataArray
                'freq': freq,
                'wave': zonalwnum,
                'nseg': result['nseg'],
                'dof': result['dof'],
                'p': result['p'],
                'prob_coh2': result['prob_coh2']
            }
            np.savez_compressed(
                os.path.join(FIG_SAVE_DIR, f'cross_spectrum_level{level}_{exp_name}.npz'),
                STC=stc,
                freq=freq,
                wave=zonalwnum,
                nseg=result['nseg'],
                dof=result['dof'],
                p=result['p'],
                prob_coh2=result['prob_coh2']
            )
            print(f"    ✓ {exp_name.upper()} 完成 (nseg={result['nseg']}, dof={result['dof']:.2f})")
        
        # ============ 步骤3: 绘制并保存图片 ============
        print(f"\n🎨 绘制 Level {level} 的交叉谱图...")
        
        s2d = 86400
        re = 6371 * 1000
        
        # 创建3个子图
        fig, axes = plt.subplots(1, 3, figsize=(16, 8), dpi=300)
        plt.subplots_adjust(left=0.06, right=0.98, top=0.92, bottom=0.15, wspace=0.25)
        plt.rcParams.update({'font.size': 10})
        
        exp_names = ['cntl', 'p4k', '4co2']
        exp_titles = ['CNTL', 'P4K', '4CO2']
        
        # 设置coherence²阈值
        coh2_threshold = 0.05
        
        for idx, (exp_name, exp_title, ax) in enumerate(zip(exp_names, exp_titles, axes)):
            plt.sca(ax)
            
            # 获取当前实验的结果
            stc = level_results[exp_name]['STC']
            zonalwnum = level_results[exp_name]['wave']
            freq = level_results[exp_name]['freq']
            
            # 绘制coherence squared的等高线图
            # 使用sel来选择component维度
            contourf = stc.sel(component='COH2').plot.contourf(
                ax=ax,
                cmap=cmaps.WhiteBlueGreenYellowRed,
                levels=21,
                add_colorbar=False,
                add_labels=False,
                extend='neither'
            )
            
            # 准备矢量场数据
            skip = 2
            wave_sub = zonalwnum[::skip]
            freq_sub = freq[::skip]
            u_sub = stc.sel(component='V1').values[::skip, ::skip]
            v_sub = stc.sel(component='V2').values[::skip, ::skip]
            coh2_sub = stc.sel(component='COH2').values[::skip, ::skip]
            
            # 创建掩蔽
            mask = coh2_sub < coh2_threshold
            u_masked = np.where(mask, np.nan, u_sub)
            v_masked = np.where(mask, np.nan, v_sub)
            
            # 添加矢量场
            q = ax.quiver(
                wave_sub, freq_sub,
                u_masked, v_masked,
                scale=30, headwidth=4, headlength=5, 
                width=0.004, alpha=0.8
            )
            
            # 设置标题
            ax.set_title(f'({chr(97 + idx)}) {exp_title}', fontsize=18, loc='left')
            ax.set_title(f'Symm | Level {level}', fontsize=10, loc='right')
            
            # 设置坐标轴
            ax.set_ylabel('Frequency (1/day)', fontsize=18)
            ax.set_xlabel('Zonal wavenumber', fontsize=18)
            ax.set_xlim(-15, 15)
            ax.set_ylim(0, 0.5)
            
            # 绘制CCKW波段
            ax.plot(kw_x[0], kw_y[0], 'purple', linewidth=1.5, linestyle='solid', label='CCKW band')
            
            # 添加参考线
            d = np.array([3, 6, 20])
            dname = ['3d', '6d', '20d']
            he_all = np.array([8, 25, 90])
            cp = (9.8 * he_all) ** 0.5
            zwnum_goal = 0.5 / s2d / cp * 2 * np.pi * re
            
            # 零波数线
            ax.plot([0, 0], [0, 0.5], 'k', linewidth=1, linestyle=':')
            
            # 周期线
            for dd in range(len(d)):
                ax.plot([-15, 15], [1/d[dd], 1/d[dd]], 'k', linewidth=1, linestyle=':')
                ax.text(-14.8, 1/d[dd] + 0.01, dname[dd], fontsize=15, color='k')
            
            # Kelvin波色散关系
            for hh in range(len(he_all)):
                ax.plot([0, zwnum_goal[hh]], [0, 0.5], 'grey', linewidth=1, linestyle='dashed')
            
            # 添加"kelvin"标签
            ax.text(12, 0.35, 'kelvin', ha="center", va="center", size=9,
                    bbox={'facecolor': 'w', 'alpha': 0.9, 'edgecolor': 'none'})
            ax.tick_params(labelsize=18, which='both', top=True, right=True)
        
        # 添加颜色条
        cbar = fig.colorbar(contourf, ax=axes, orientation='horizontal', 
                             pad=0.15, aspect=40, shrink=0.8)
        cbar.set_label('Coherence squared', fontsize=11)
        
        # 保存图片
        save_path = os.path.join(FIG_SAVE_DIR, f'cross_spectrum_lev{level}.png')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"    ✅ 图片已保存: {save_path}")
        
        plt.close(fig)  # 关闭图形以释放内存
        
        print(f"\n✅ Level {level} 处理完成！")
        
    except Exception as e:
        print(f"\n❌ Level {level} 处理失败: {e}")
        import traceback
        traceback.print_exc()
        continue

print(f"\n{'='*80}")
print(f"🎉 所有层次处理完成！")
print(f"{'='*80}")


🔄 处理 Level 41

📂 加载 Level 41 的散度数据...
📂 加载数据: /work/mh1498/m301257/3D_data/divergence/divergence_lev41_cntl.nc
✅ 数据加载成功
   形状: (5114, 15, 180)
   维度: ('time', 'lat', 'lon')
   时间范围: 1980-01-01T00:00:00.000000000 to 1993-12-31T00:00:00.000000000
   空间范围: lat [-14.0, 14.0], lon [0.0, 358.0]
📂 加载数据: /work/mh1498/m301257/3D_data/divergence/divergence_lev41_p4k.nc
✅ 数据加载成功
   形状: (5114, 15, 180)
   维度: ('time', 'lat', 'lon')
   时间范围: 1980-01-01T00:00:00.000000000 to 1993-12-31T00:00:00.000000000
   空间范围: lat [-14.0, 14.0], lon [0.0, 358.0]
📂 加载数据: /work/mh1498/m301257/3D_data/divergence/divergence_lev41_4co2.nc
✅ 数据加载成功
   形状: (5114, 15, 180)
   维度: ('time', 'lat', 'lon')
   时间范围: 1980-01-01T00:00:00.000000000 to 1993-12-31T00:00:00.000000000
   空间范围: lat [-14.0, 14.0], lon [0.0, 358.0]

📊 计算 Level 41 的交叉谱...

  处理 CNTL...
  原始形状: pr=(5114, 15, 180), div=(5114, 15, 180)
  对齐后形状: pr=(5114, 15, 180), div=(5114, 15, 180)
    ✓ CNTL 完成 (nseg=168, dof=448.06)

  处理 P4K...
  原始形状: pr=(5114, 15, 

In [18]:
# ============ 查看生成的所有图片 ============

print("生成的图片文件:")
print("="*80)

generated_files = sorted(glob.glob(os.path.join(FIG_SAVE_DIR, 'cross_spectrum_lev*.png')))

if generated_files:
    for i, file in enumerate(generated_files, 1):
        file_size = os.path.getsize(file) / (1024 * 1024)  # MB
        print(f"{i:2d}. {os.path.basename(file):40s} ({file_size:.2f} MB)")
    print(f"\n✅ 共生成 {len(generated_files)} 张图片")
    print(f"📁 保存目录: {FIG_SAVE_DIR}")
else:
    print("⚠️  没有找到生成的图片文件")
    
print("="*80)

生成的图片文件:
 1. cross_spectrum_lev41.png                 (0.64 MB)
 2. cross_spectrum_lev46.png                 (0.90 MB)
 3. cross_spectrum_lev51.png                 (0.87 MB)
 4. cross_spectrum_lev55.png                 (0.72 MB)
 5. cross_spectrum_lev58.png                 (0.65 MB)
 6. cross_spectrum_lev63.png                 (0.59 MB)
 7. cross_spectrum_lev67.png                 (0.54 MB)
 8. cross_spectrum_lev71.png                 (0.52 MB)
 9. cross_spectrum_lev74.png                 (0.60 MB)
10. cross_spectrum_lev76.png                 (0.67 MB)
11. cross_spectrum_lev78.png                 (0.63 MB)
12. cross_spectrum_lev80.png                 (0.73 MB)
13. cross_spectrum_lev81.png                 (0.95 MB)
14. cross_spectrum_lev83.png                 (1.16 MB)
15. cross_spectrum_lev84.png                 (1.15 MB)
16. cross_spectrum_lev85.png                 (1.12 MB)
17. cross_spectrum_lev87.png                 (1.10 MB)

✅ 共生成 17 张图片
📁 保存目录: ../figures/cross_spectrum_specific